In [ ]:
"""
Effectiveness of thrombolysis and thrombectomy in early (<6hr) arrivals of
non-mild stroke - discharge mRS 0-2.

CAUSAL FOREST version (EconML CausalForestDML).

This is a re-implementation of the S-learner counterfactual notebook
(`01_thrombolysis_thrombectomy_mRS_2`) as an *orthogonalized* causal forest
(Double ML / R-learner). It keeps the same cohort, features and 5-fold
cross-fitting spirit, but replaces the "flip the treatment feature to a 99999
sentinel and re-predict" S-learner with a doubly-robust causal forest that
provides honest CATE confidence intervals.

Two estimands are produced, because section 1.3 of the original states the key
features of interest are the *times*, while the grouped analysis is built around
treatment *strategy*:

  A. STRATEGY (multi-arm, discrete):
       arms = {neither, thrombolysis_only, thrombectomy_only, both}
       baseline arm = "neither"
       -> CATE of each strategy vs no treatment, on the RISK (probability) scale.

  B. TIMING (continuous dose-response, treated-only):
       For each modality (thrombolysis, thrombectomy) fit a continuous-treatment
       causal forest on the treated subgroup only (NO sentinel), giving the
       average partial effect of *delay* (per minute, and reported per hour) on
       P(good outcome), with heterogeneity by X.

Notes on design choices (see accompanying discussion):
  * The 99999 sentinel is REMOVED. A causal forest cannot use a point-mass +
    continuum mixture as a treatment.
  * Outcome is binary; effects are on the probability (risk-difference) scale
    via discrete_outcome=True with a classifier model_y. This differs from the
    log-odds contrasts in the S-learner notebook (documented, not a bug).
  * stroke_team is high-cardinality; it is passed as `groups` for cluster-robust
    cross-fitting rather than one-hot encoded, and is NOT used as a heterogeneity
    (X) feature. Its confounding role is still captured because it is included in
    the nuisance control set W via a frequency/target-style ordinal encoding.
  * ethnicity is ordinal-encoded for the nuisance models.

Requires: econml>=0.16, lightgbm, shap, scikit-learn, pandas, numpy.
"""

In [ ]:
import os
import warnings

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
import matplotlib
# Use an interactive-friendly backend so figures render in the notebook.
# (The original "Agg" backend is headless and never displays inline.)
try:
    get_ipython()  # noqa: F821  -- defined only inside IPython/Jupyter
    _IN_NOTEBOOK = True
except NameError:
    _IN_NOTEBOOK = False
    matplotlib.use("Agg")  # headless fallback for plain `python script.py`
import matplotlib.pyplot as plt
if _IN_NOTEBOOK:
    try:
        get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
    except Exception:
        pass


In [ ]:
from econml.dml import CausalForestDML

In [ ]:
warnings.filterwarnings("ignore")
# Silence EconML/joblib progress bars; set VERBOSE_FIT=1 to re-enable.
_VERBOSE = int(os.environ.get("VERBOSE_FIT", "0"))
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [ ]:
# ---------------------------------------------------------------------------
# 1. Configuration -- identical cohort / features to the S-learner notebook
# ---------------------------------------------------------------------------
DATA_PATH = "../../data/sam3/cleaned_data.csv"
MRS_TARGET = 2  # good outcome = discharge_disability <= 2
MIN_THROMBECTOMY = 10  # keep only stroke teams with >= this many thrombectomy cases

In [ ]:
# --- Fast-testing option: balanced bootstrap resample -----------------------
# When BOOTSTRAP_BALANCED is True, draw BOOTSTRAP_N rows WITH REPLACEMENT from
# each of the four treatment groups, giving a balanced 4 x N working sample.
# This is a SPEED / DEBUG aid only -- see the docstring of
# balanced_bootstrap_sample() for why it changes what is being estimated.
BOOTSTRAP_BALANCED = True
BOOTSTRAP_N = 1000

In [ ]:
# Adjustment / control features (same list as the original notebook).
# These act as BOTH heterogeneity features X (clinical, pre-treatment) and,
# together with the encoded categoricals, as nuisance controls W.
HETERO_FEATURES = [
    "prior_disability",
    "stroke_severity",
    "age",
    "congestive_heart_failure",
    "hypertension",
    "diabetes",
    "afib_anticoagulant",
    "any_afib_diagnosis",
]
CATEGORICAL_FEATURES = ["stroke_team", "ethnicity"]

In [ ]:
TIME_THROMBOLYSIS = "onset_to_thrombolysis_time"
TIME_THROMBECTOMY = "onset_to_thrombectomy_time"

In [ ]:
# ---------------------------------------------------------------------------
# 2. Load + filter -- byte-for-byte the same selection logic as the notebook
# ---------------------------------------------------------------------------
def load_and_filter(path=DATA_PATH):
    data = pd.read_csv(path, low_memory=False)

    for col in CATEGORICAL_FEATURES:
        data[col] = data[col].replace("", "Empty").astype("category")
    numeric_cols = HETERO_FEATURES + [TIME_THROMBOLYSIS, TIME_THROMBECTOMY]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    print(f"Initial data shape: {data.shape}")

    data = data.dropna(subset=["discharge_disability"])
    data = data[data["infarction"] == 1]
    data = data[data["onset_to_arrival_time"] < 360]
    data = data[data["perfusion_imaging_used"] == 0]
    data = data[((data["stroke_severity"] > 5) | (data["lvo"] == 1))]

    # Missing treatment times stay as NaN (no magic sentinel); NaN == "did not
    # receive this treatment". All comparisons below are NaN-aware.
    both = data[TIME_THROMBOLYSIS].notna() & data[TIME_THROMBECTOMY].notna()
    lysis_after_ectomy = data[TIME_THROMBOLYSIS] > data[TIME_THROMBECTOMY]
    data = data[~(both & lysis_after_ectomy)]

    data = data[(data[TIME_THROMBECTOMY] < 720) | data[TIME_THROMBECTOMY].isna()]
    data = data[(data[TIME_THROMBOLYSIS] < 360) | data[TIME_THROMBOLYSIS].isna()]

    # Keep only stroke teams that performed at least MIN_THROMBECTOMY
    # thrombectomies (drop all rows from lower-volume teams). A team's
    # thrombectomy count = rows with a non-missing thrombectomy time.
    ect_counts = (
        data.loc[data[TIME_THROMBECTOMY].notna(), "stroke_team"]
        .value_counts()
    )
    keep_teams = ect_counts[ect_counts >= MIN_THROMBECTOMY].index
    n_teams_before = data["stroke_team"].nunique()
    data = data[data["stroke_team"].isin(keep_teams)]
    # Drop now-unused stroke_team categories so encodings stay tidy.
    if str(data["stroke_team"].dtype) == "category":
        data["stroke_team"] = data["stroke_team"].cat.remove_unused_categories()
    print(f"Stroke-team thrombectomy filter (>= {MIN_THROMBECTOMY}): "
          f"{n_teams_before} -> {len(keep_teams)} teams kept")

    print(f"Final data shape: {data.shape}")
    return data.reset_index(drop=True)

In [ ]:
def add_treatment_columns(data):
    """Derive clean binary indicators + a 4-level strategy arm.

    A treatment is "received" iff its onset-to-treatment time is present
    (non-NaN); missing time means the patient did not get that treatment.
    """
    got_lysis = data[TIME_THROMBOLYSIS].notna().astype(int)
    got_ectomy = data[TIME_THROMBECTOMY].notna().astype(int)
    data = data.copy()
    data["got_thrombolysis"] = got_lysis
    data["got_thrombectomy"] = got_ectomy

    def strategy(row):
        if row["got_thrombolysis"] and row["got_thrombectomy"]:
            return "both"
        if row["got_thrombolysis"]:
            return "thrombolysis_only"
        if row["got_thrombectomy"]:
            return "thrombectomy_only"
        return "neither"

    data["treatment_group"] = data.apply(strategy, axis=1)
    return data

In [ ]:
# ---------------------------------------------------------------------------
# 2b. Balanced bootstrap resample (FAST-TESTING ONLY)
# ---------------------------------------------------------------------------
def balanced_bootstrap_sample(data, n_per_group=BOOTSTRAP_N, seed=RANDOM_STATE):
    """
    Draw `n_per_group` rows WITH REPLACEMENT from each treatment_group
    (neither / thrombolysis_only / thrombectomy_only / both), returning a
    balanced 4 x n_per_group frame for faster experimentation.

    IMPORTANT -- this changes the estimand, do NOT use for final estimates:
      * The natural treatment prevalences (your propensity distribution) are
        destroyed. A DML causal forest residualises treatment on X to estimate
        e(x)=P(T|X); resampling to equal arm sizes forces the MARGINAL
        propensities to 1/4 each. Effects are then implicitly reweighted toward
        the rare arms (e.g. `both`, thrombectomy), so the ATE it reports is no
        longer the population ATE -- it drifts toward an unweighted average of
        arm-specific effects over an artificial population.
      * Sampling with replacement duplicates patients. Duplicates that land in
        different cross-fitting folds leak information across the train/eval
        split, breaking honesty and making confidence intervals too NARROW
        (over-optimistic). Cluster-robust CIs via `groups` partly mitigate this
        only if duplicates share a cluster id -- here they share stroke_team,
        not a patient id, so leakage is not fully controlled.
      * Point estimates of *within-arm* contrasts and the SHAPE of heterogeneity
        (BLP signs, which features modify effects) are usually still directionally
        fine -- which is exactly what you want for a fast smoke test.

    Use this to iterate on code / runtime, then set BOOTSTRAP_BALANCED=False for
    the real fit on the full cohort.
    """
    rng_local = np.random.default_rng(seed)
    counts = data["treatment_group"].value_counts().to_dict()
    parts = []
    print(f"\n[bootstrap] balanced resample: {n_per_group} per group "
          f"(with replacement)")
    for grp in ["neither", "thrombolysis_only", "thrombectomy_only", "both"]:
        pool = data[data["treatment_group"] == grp]
        if len(pool) == 0:
            print(f"  [warn] group '{grp}' is empty -- skipped")
            continue
        idx = rng_local.integers(0, len(pool), size=n_per_group)
        parts.append(pool.iloc[idx])
        print(f"  {grp:<20s}: {len(pool):>6d} available -> "
              f"{n_per_group} sampled ({n_per_group / max(len(pool),1):.1f}x)")
    out = pd.concat(parts, axis=0).reset_index(drop=True)
    print(f"[bootstrap] working sample shape: {out.shape}")
    return out

In [ ]:
# ---------------------------------------------------------------------------
# 3pre. Categorical encoding helper for NATIVE LightGBM categorical handling
# ---------------------------------------------------------------------------
def encode_categoricals_as_codes(frame, cat_cols):
    """
    Turn nominal categorical columns into NON-NEGATIVE integer codes suitable
    for LightGBM's native categorical handling (passed via `categorical_feature`
    as column indices). Unknown/missing map to a dedicated highest code rather
    than -1 (LightGBM requires non-negative category ids). Returns an (n, k)
    int array with one column per cat_col, in the given order.
    """
    codes = []
    for col in cat_cols:
        ser = frame[col].astype(str).replace({"nan": "Empty", "": "Empty"}).fillna("Empty")
        cat = ser.astype("category").cat.codes.to_numpy()   # 0..K-1, no -1 here
        codes.append(cat.astype(np.int64))
    return np.column_stack(codes) if codes else np.empty((len(frame), 0))


In [ ]:
# ---------------------------------------------------------------------------
# 3. Feature matrices
# ---------------------------------------------------------------------------
def build_matrices(data):
    """
    Returns:
      X  : heterogeneity features (pre-treatment clinical only) for CATE splits
      W  : nuisance controls (X + integer-coded categoricals) for residualising
      cat_idx : column indices in W that are categorical (for LightGBM)
      y  : binary good-outcome target
      groups : stroke_team codes for cluster-robust cross-fitting
    """
    y = (data["discharge_disability"] <= MRS_TARGET).astype(int).to_numpy()

    # Heterogeneity features: clinical, pre-treatment, interpretable.
    X = data[HETERO_FEATURES].copy()
    # Median-impute any residual NaNs (e.g. stroke_severity had a few missing).
    X = X.fillna(X.median(numeric_only=True))

    # Nuisance controls W = X plus encoded categoricals so treatment/outcome
    # models can adjust for hospital and ethnicity confounding.
    # Integer-code the nominal categoricals (stroke_team, ethnicity) so
    # LightGBM can split on them NATIVELY as categories (not as ordinals).
    cat_encoded = encode_categoricals_as_codes(data, CATEGORICAL_FEATURES)
    n_num = X.shape[1]
    cat_idx = list(range(n_num, n_num + cat_encoded.shape[1]))
    W = np.hstack([X.to_numpy(), cat_encoded])
    assert not np.isnan(W).any(), "W (nuisance controls) still contains NaN"

    # Cluster groups for honest, cluster-robust inference.
    # If a bootstrap patient id exists, cluster by ORIGINAL patient so that
    # duplicated rows are kept in the same cross-fitting fold (mitigates the
    # information leakage from sampling with replacement). Otherwise cluster by
    # stroke_team (the natural hospital-level correlation structure).
    if "_patient_id" in data.columns:
        groups = data["_patient_id"].to_numpy()
    else:
        groups = data["stroke_team"].astype(str).astype("category").cat.codes.to_numpy()

    return X, W, y, groups, cat_idx


In [ ]:
# ---------------------------------------------------------------------------
# Picklable LightGBM wrappers with NATIVE categorical handling.
# LightGBM (>=4.x) IGNORES `categorical_feature` on the constructor, so we
# inject it into `.fit()`. These are defined at MODULE LEVEL (not inside a
# function) so fitted EconML estimators containing them remain picklable and
# `sklearn.clone`-compatible. `cat_features` is a real constructor param, so
# get_params/set_params round-trips it (required by clone / EconML).
# ---------------------------------------------------------------------------
class CatLGBMClassifier(LGBMClassifier):
    def __init__(self, cat_features="auto", **kwargs):
        self.cat_features = cat_features
        super().__init__(**kwargs)

    def get_params(self, deep=True):
        params = super().get_params(deep=deep)
        params["cat_features"] = self.cat_features
        return params

    def fit(self, X, y, **kw):
        kw.setdefault("categorical_feature", self.cat_features)
        return super().fit(X, y, **kw)


class CatLGBMRegressor(LGBMRegressor):
    def __init__(self, cat_features="auto", **kwargs):
        self.cat_features = cat_features
        super().__init__(**kwargs)

    def get_params(self, deep=True):
        params = super().get_params(deep=deep)
        params["cat_features"] = self.cat_features
        return params

    def fit(self, X, y, **kw):
        kw.setdefault("categorical_feature", self.cat_features)
        return super().fit(X, y, **kw)


def make_nuisances(discrete_treatment, cat_features=None):
    """LightGBM nuisance learners with NATIVE categorical handling.

    cat_features: list of integer COLUMN INDICES in W to treat as native
    LightGBM categoricals (e.g. stroke_team, ethnicity). Passed to the
    picklable CatLGBM* wrappers, which inject it into `.fit()`.
    Codes must be non-negative integers (see encode_categoricals_as_codes).
    """
    cat = cat_features if cat_features is not None else "auto"
    common = dict(
        n_estimators=400, num_leaves=31, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=-1,
    )
    model_y = CatLGBMClassifier(cat_features=cat, **common)
    if discrete_treatment:
        model_t = CatLGBMClassifier(cat_features=cat, **common)
    else:
        model_t = CatLGBMRegressor(cat_features=cat, **common)
    return model_y, model_t


In [ ]:
# ---------------------------------------------------------------------------
# 4A. STRATEGY analysis -- multi-arm discrete causal forest vs "neither"
# ---------------------------------------------------------------------------
def run_strategy_forest(data, X, W, y, groups, cat_idx=None):
    print("\n" + "=" * 70)
    print("A. STRATEGY (multi-arm discrete): each arm vs 'neither'")
    print("=" * 70)

    # EconML one-hot encodes a discrete treatment internally; the alphabetically
    # first level is the baseline. We force "neither" to sort first.
    arm = data["treatment_group"].map(
        {"neither": "0_neither",
         "thrombolysis_only": "1_thrombolysis_only",
         "thrombectomy_only": "2_thrombectomy_only",
         "both": "3_both"}
    ).to_numpy()

    model_y, model_t = make_nuisances(discrete_treatment=True, cat_features=cat_idx)
    est = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        discrete_outcome=True,           # binary mRS good-outcome target
        n_estimators=2000,
        min_samples_leaf=25,
        max_depth=None,
        honest=True,
        cv=5,                            # 5-fold cross-fitting (matches notebook)
        verbose=_VERBOSE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    est.fit(y, arm, X=X.to_numpy(), W=W, groups=groups)

    baseline = "0_neither"
    other_arms = ["1_thrombolysis_only", "2_thrombectomy_only", "3_both"]

    print("\nDoubly-robust ATE of each strategy vs 'neither' "
          "(risk-difference in P(mRS 0-2)):")
    results = {}
    for a in other_arms:
        ate = est.ate(X.to_numpy(), T0=baseline, T1=a)
        lb, ub = est.ate_interval(X.to_numpy(), T0=baseline, T1=a, alpha=0.05)
        cate = est.effect(X.to_numpy(), T0=baseline, T1=a)
        results[a] = cate
        print(f"  {a[2:]:<20s}: ATE = {float(np.ravel(ate)[0]):+.4f} "
              f"[95% CI {float(np.ravel(lb)[0]):+.4f}, {float(np.ravel(ub)[0]):+.4f}]  "
              f"| CATE mean={cate.mean():+.4f} sd={cate.std():.4f}")

    # SHAP on the CATE model, for one contrast (thrombolysis_only vs neither).
    try:
        shap_values = est.shap_values(X.to_numpy(), feature_names=list(X.columns))
        print("\nSHAP values computed for the strategy CATE model "
              "(keys):", list(shap_values.keys()))
    except Exception as exc:  # SHAP is best-effort
        print(f"\n[warn] SHAP for strategy model skipped: {exc}")

    return est, results

In [ ]:
# ---------------------------------------------------------------------------
# 4B. TIMING analysis -- continuous dose-response, treated subgroup only
# ---------------------------------------------------------------------------
def run_timing_forest(data, modality):
    """
    modality: 'thrombolysis' or 'thrombectomy'.
    Fit a continuous-treatment causal forest on the TREATED subgroup only.
    Treatment W = onset-to-treatment time (minutes). Untreated (NaN) excluded.
    Estimand: average partial effect (per-minute change in P(good outcome));
    reported per hour of additional delay.
    """
    print("\n" + "=" * 70)
    print(f"B. TIMING (continuous dose-response): {modality} treated-only")
    print("=" * 70)

    time_col = TIME_THROMBOLYSIS if modality == "thrombolysis" else TIME_THROMBECTOMY
    treated = data[data[time_col].notna()].copy()
    print(f"  Treated subgroup n = {len(treated)}")

    X = treated[HETERO_FEATURES].copy()
    X = X.fillna(X.median(numeric_only=True))

    # Native LightGBM categorical codes for stroke_team / ethnicity.
    cat_encoded = encode_categoricals_as_codes(treated, CATEGORICAL_FEATURES)
    n_num = X.shape[1]
    cat_idx = list(range(n_num, n_num + cat_encoded.shape[1]))
    W = np.hstack([X.to_numpy(), cat_encoded])
    assert not np.isnan(W).any(), "W (nuisance controls) still contains NaN"

    T = treated[time_col].to_numpy().astype(float)  # minutes, continuous
    y = (treated["discharge_disability"] <= MRS_TARGET).astype(int).to_numpy()
    if "_patient_id" in treated.columns:
        groups = treated["_patient_id"].to_numpy()
    else:
        groups = treated["stroke_team"].astype(str).astype("category").cat.codes.to_numpy()

    model_y, model_t = make_nuisances(discrete_treatment=False, cat_features=cat_idx)
    est = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=False,
        discrete_outcome=True,
        n_estimators=2000,
        min_samples_leaf=25,
        honest=True,
        cv=5,
        verbose=_VERBOSE,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    est.fit(y, T, X=X.to_numpy(), W=W, groups=groups)

    # Constant marginal (per-minute) effect and per-hour interpretation.
    cme = est.const_marginal_effect(X.to_numpy())          # per minute, per patient
    lb, ub = est.const_marginal_effect_interval(X.to_numpy(), alpha=0.05)
    per_min = float(np.mean(cme))
    print(f"  Average partial effect: {per_min:+.6f} per minute "
          f"= {per_min * 60:+.4f} per HOUR of delay in P(mRS 0-2)")
    print(f"    (per-minute 95% CI mean: [{float(np.mean(lb)):+.6f}, "
          f"{float(np.mean(ub)):+.6f}])")
    print(f"  CATE heterogeneity across patients (per hour): "
          f"sd={cme.std() * 60:.4f}")

    # Return X (treated heterogeneity features) and T for plotting.
    return est, X, T

In [ ]:
# ---------------------------------------------------------------------------
# 4C. Best Linear Projection (BLP) of the CATE onto interpretable features
# ---------------------------------------------------------------------------
def best_linear_projection(est, X, label, feature_names, T0=None, T1=None):
    """
    Best linear projection of the CATE onto X: the OLS fit of tau(X) on the
    heterogeneity features (with a doubly-robust score), giving a coefficient +
    95% CI per feature. A significant coefficient means the treatment effect
    varies with that covariate. This is the honest analogue of reading a SHAP
    dependence plot: it is a *test* of effect modification, not just importance.

    See Semenova & Chernozhukov (2021). We project the estimated CATE onto the
    heterogeneity features by OLS with HC1 heteroskedasticity-robust standard
    errors. (EconML's `cate_interpreter` / linear BLP helpers are not exposed for
    CausalForestDML with custom projection features, so this explicit OLS on the
    fitted tau(X) is the transparent equivalent.)
    """
    print("\n" + "-" * 70)
    print(f"Best Linear Projection of CATE: {label}")
    print("-" * 70)
    Xa = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)

    if T0 is not None:
        tau = np.ravel(est.effect(Xa, T0=T0, T1=T1))
    else:
        tau = np.ravel(est.const_marginal_effect(Xa))

    # Standardise features so coefficients are comparable per 1-SD change.
    mu, sigma = Xa.mean(axis=0), Xa.std(axis=0)
    sigma[sigma == 0] = 1.0
    Xz = (Xa - mu) / sigma
    Xd = np.column_stack([np.ones(len(Xz)), Xz])
    beta, *_ = np.linalg.lstsq(Xd, tau, rcond=None)
    resid = tau - Xd @ beta
    n, k = Xd.shape
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    meat = Xd.T @ (Xd * (resid ** 2)[:, None])          # HC1 sandwich
    cov = XtX_inv @ meat @ XtX_inv * (n / (n - k))
    se = np.sqrt(np.diag(cov))

    names = ["intercept"] + list(feature_names)
    print(f"{'feature':<26s}{'coef/SD':>12s}{'se':>10s}{'z':>8s}"
          f"{'[0.025':>12s}{'0.975]':>12s}")
    for nm, b, s in zip(names, beta, se):
        z = b / s if s > 0 else 0.0
        star = "*" if abs(z) > 1.96 else " "
        print(f"{nm:<26s}{b:>12.5f}{s:>10.5f}{z:>8.2f}"
              f"{b - 1.96*s:>12.5f}{b + 1.96*s:>12.5f} {star}")
    print("  intercept = mean effect; other coefs = change per +1 SD of feature.")
    print("  (* = |z| > 1.96, i.e. significant effect modification at 5%)")
    return dict(zip(names, beta))

In [ ]:
# ---------------------------------------------------------------------------
# 5. Optional cross-check: how well do the nuisance outcome models fit?
# ---------------------------------------------------------------------------
def nuisance_sanity_check(X, W, y, groups, cat_idx=None):
    """Quick AUC / balanced accuracy of the outcome nuisance E[Y|W]."""
    from sklearn.model_selection import cross_val_predict
    from sklearn.model_selection import StratifiedGroupKFold

    model_y, _ = make_nuisances(discrete_treatment=True, cat_features=cat_idx)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    proba = cross_val_predict(
        model_y, W, y, cv=cv, groups=groups, method="predict_proba", n_jobs=-1
    )[:, 1]
    auc = roc_auc_score(y, proba)
    bal = balanced_accuracy_score(y, (proba >= y.mean()).astype(int))
    print(f"\n[nuisance check] outcome model E[Y|W]: AUC={auc:.4f}, "
          f"balanced acc={bal:.4f}")

## Visualization suite

All figures below display inline and are saved to the `output/` folder next to this notebook (PNG + PDF). Sections mirror the analysis:
A = strategy CATE, B = effect modification, C = timing dose-response, D = model diagnostics.

In [ ]:
# ---------------------------------------------------------------------------
# 4F. Plotting configuration & helpers
# ---------------------------------------------------------------------------
from pathlib import Path
from scipy.special import expit, logit

# `output/` lives next to the notebook. In a notebook the CWD is the notebook
# directory; when run as a script we fall back to this file's location.
try:
    _BASE_DIR = Path(__file__).resolve().parent
except NameError:
    _BASE_DIR = Path.cwd()
OUTPUT_DIR = _BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Every saved artifact (figures AND data) is tagged with the mRS threshold so
# runs at different MRS_TARGET values do not overwrite each other.
MRS_TAG = f"mrs{MRS_TARGET}"
print(f"[plots] figures + data will be saved to: {OUTPUT_DIR} (tag: {MRS_TAG})")


def out_path(filename):
    """Absolute path inside output/ with the MRS_TARGET tag prefixed."""
    return OUTPUT_DIR / f"{MRS_TAG}_{filename}"

# Shared, publication-friendly styling (works for LaTeX/REF figures).
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

# Consistent colour per treatment arm across every figure.
ARM_COLORS = {
    "thrombolysis_only": "#1f77b4",
    "thrombectomy_only": "#ff7f0e",
    "both": "#2ca02c",
    "neither": "#7f7f7f",
}
ARM_LABELS = {
    "thrombolysis_only": "Thrombolysis only",
    "thrombectomy_only": "Thrombectomy only",
    "both": "Both",
    "neither": "Neither",
}
MODALITY_COLORS = {"thrombolysis": "#1f77b4", "thrombectomy": "#ff7f0e"}


def save_and_show(fig, name):
    """Save a figure to OUTPUT_DIR (PNG + PDF, MRS-tagged) and display inline."""
    png = out_path(f"{name}.png")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(out_path(f"{name}.pdf"), bbox_inches="tight")
    print(f"[plots] saved -> {png}")
    plt.show()
    plt.close(fig)


In [ ]:
# ---------------------------------------------------------------------------
# A1. ATE forest plot: each strategy vs "neither" (risk difference)
# ---------------------------------------------------------------------------
def plot_ate_forest(est, X, name="A1_ate_forest_strategy"):
    baseline = "0_neither"
    arms = [("1_thrombolysis_only", "thrombolysis_only"),
            ("2_thrombectomy_only", "thrombectomy_only"),
            ("3_both", "both")]
    Xa = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)

    rows = []
    for code, key in arms:
        ate = float(np.ravel(est.ate(Xa, T0=baseline, T1=code))[0])
        lb, ub = est.ate_interval(Xa, T0=baseline, T1=code, alpha=0.05)
        rows.append((key, ate, float(np.ravel(lb)[0]), float(np.ravel(ub)[0])))

    fig, ax = plt.subplots(figsize=(8, 3.4))
    ypos = np.arange(len(rows))[::-1]
    for y, (key, ate, lb, ub) in zip(ypos, rows):
        c = ARM_COLORS[key]
        ax.plot([lb, ub], [y, y], color=c, lw=2.5, solid_capstyle="round")
        ax.plot(ate, y, "o", color=c, ms=9, zorder=3)
        ax.annotate(f"{ate:+.3f}  [{lb:+.3f}, {ub:+.3f}]",
                    (ub, y), xytext=(8, 0), textcoords="offset points",
                    va="center", fontsize=9)
    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.7)
    ax.set_yticks(ypos)
    ax.set_yticklabels([ARM_LABELS[k] for k, *_ in rows])
    ax.set_xlabel("ATE: change in P(mRS 0-2) vs no treatment (risk difference)")
    ax.set_title("Strategy effect on good outcome (95% CI)")
    ax.margins(x=0.28)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# A2. Per-patient CATE distributions per strategy (heterogeneity)
# ---------------------------------------------------------------------------
def plot_cate_distributions(strat_cate, name="A2_cate_distributions"):
    from scipy.stats import gaussian_kde
    key_of = {"1_thrombolysis_only": "thrombolysis_only",
              "2_thrombectomy_only": "thrombectomy_only",
              "3_both": "both"}
    fig, ax = plt.subplots(figsize=(9, 4.5))
    allvals = np.concatenate([np.ravel(v) for v in strat_cate.values()])
    grid = np.linspace(allvals.min(), allvals.max(), 300)
    for code, cate in strat_cate.items():
        key = key_of[code]
        vals = np.ravel(cate)
        c = ARM_COLORS[key]
        try:
            dens = gaussian_kde(vals)(grid)
            ax.fill_between(grid, dens, alpha=0.25, color=c)
            ax.plot(grid, dens, color=c, lw=2,
                    label=f"{ARM_LABELS[key]} (mean {vals.mean():+.3f}, sd {vals.std():.3f})")
        except Exception:
            ax.hist(vals, bins=40, density=True, alpha=0.4, color=c,
                    label=ARM_LABELS[key])
        ax.axvline(vals.mean(), color=c, ls=":", lw=1.2)
    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.7)
    ax.set_xlabel("Per-patient CATE: change in P(mRS 0-2) vs neither")
    ax.set_ylabel("Density")
    ax.set_title("Heterogeneity of treatment effect across patients")
    ax.legend(fontsize=8.5)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# A3. SHAP summary (beeswarm) for the strategy CATE model
# ---------------------------------------------------------------------------
def plot_shap_summary(est, X, name="A3_shap_summary"):
    try:
        import shap
    except Exception as exc:
        print(f"[plots] SHAP not available, skipping beeswarm: {exc}")
        return
    Xa = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
    feat_names = list(X.columns) if hasattr(X, "columns") else \
        [f"x{i}" for i in range(Xa.shape[1])]
    try:
        shap_values = est.shap_values(Xa, feature_names=feat_names)
    except Exception as exc:
        print(f"[plots] est.shap_values failed, skipping: {exc}")
        return

    # shap_values is nested {output_name: {treatment_name: Explanation}}.
    # The treatment_name (e.g. "T0_3_both") identifies the contrast vs the
    # baseline arm ("neither"), so we use it to label each plot meaningfully.
    def _named_leaves(d):
        for k, v in d.items():
            if isinstance(v, dict):
                for kk, vv in _named_leaves(v):
                    yield (kk, vv)  # inner (treatment) key wins as the label
            else:
                yield (k, v)

    def _pretty_contrast(key):
        # Strip EconML's "T0_<idx>_" prefix -> arm name -> "<Arm> vs neither".
        arm = key
        if arm.startswith("T0_"):
            arm = arm[3:]
        # drop a leading "<digit>_" ordering prefix if present
        if len(arm) > 2 and arm[0].isdigit() and arm[1] == "_":
            arm = arm[2:]
        label = ARM_LABELS.get(arm, arm.replace("_", " ").title())
        return label, arm

    leaves = list(_named_leaves(shap_values))
    if not leaves:
        print("[plots] no SHAP explanations returned, skipping")
        return
    for key, expl in leaves:
        label, arm = _pretty_contrast(key)
        fig = plt.figure(figsize=(8, 4.5))
        try:
            shap.plots.beeswarm(expl, show=False, max_display=len(feat_names))
        except Exception:
            shap.summary_plot(expl.values, features=Xa,
                              feature_names=feat_names, show=False)
        plt.title(f"SHAP: drivers of CATE heterogeneity\n({label} vs neither)")
        fig = plt.gcf()
        fig.tight_layout()
        save_and_show(fig, f"{name}_{arm}_vs_neither")


In [ ]:
# ---------------------------------------------------------------------------
# B1. Best Linear Projection coefficient plot (effect modification)
# ---------------------------------------------------------------------------
def _blp_coefs(est, X, feature_names, T0=None, T1=None):
    """Return (names, beta, se) for the BLP of tau(X) on standardised X."""
    Xa = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
    if T0 is not None:
        tau = np.ravel(est.effect(Xa, T0=T0, T1=T1))
    else:
        tau = np.ravel(est.const_marginal_effect(Xa))
    mu, sigma = Xa.mean(axis=0), Xa.std(axis=0)
    sigma[sigma == 0] = 1.0
    Xz = (Xa - mu) / sigma
    Xd = np.column_stack([np.ones(len(Xz)), Xz])
    beta, *_ = np.linalg.lstsq(Xd, tau, rcond=None)
    resid = tau - Xd @ beta
    n, k = Xd.shape
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    meat = Xd.T @ (Xd * (resid ** 2)[:, None])
    cov = XtX_inv @ meat @ XtX_inv * (n / (n - k))
    se = np.sqrt(np.diag(cov))
    return (["intercept"] + list(feature_names)), beta, se


def plot_blp_forest(strat_est, X, name="B1_blp_coefficients"):
    contrasts = [("1_thrombolysis_only", "Thrombolysis only"),
                 ("2_thrombectomy_only", "Thrombectomy only"),
                 ("3_both", "Both")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, (code, title) in zip(axes, contrasts):
        names, beta, se = _blp_coefs(strat_est, X, list(X.columns),
                                     T0="0_neither", T1=code)
        # Drop the intercept for the modifier plot (it's the mean effect).
        feats = names[1:]
        b = beta[1:]
        s = se[1:]
        ypos = np.arange(len(feats))[::-1]
        for y, bi, si in zip(ypos, b, s):
            sig = abs(bi / si) > 1.96 if si > 0 else False
            col = ARM_COLORS[title.lower().replace(" ", "_")] if sig else "#bbbbbb"
            ax.plot([bi - 1.96*si, bi + 1.96*si], [y, y], color=col, lw=2.2)
            ax.plot(bi, y, "o", color=col, ms=6)
        ax.axvline(0, color="k", lw=1, ls="--", alpha=0.7)
        ax.set_yticks(ypos)
        ax.set_yticklabels(feats, fontsize=9)
        ax.set_title(f"{title}\nvs neither")
        ax.set_xlabel("BLP coef per +1 SD")
    fig.suptitle("Effect modification: how CATE varies with covariates "
                 "(coloured = significant at 5%)", fontsize=13, y=1.02)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# B2. CATE vs covariate (shape of effect modification) for top modifiers
# ---------------------------------------------------------------------------
def plot_cate_vs_covariate(strat_est, X, top_features=("stroke_severity",
                           "prior_disability", "age"),
                           name="B2_cate_vs_covariate"):
    Xa = X.to_numpy()
    cols = list(X.columns)
    contrasts = [("1_thrombolysis_only", "thrombolysis_only"),
                 ("2_thrombectomy_only", "thrombectomy_only"),
                 ("3_both", "both")]
    feats = [f for f in top_features if f in cols]
    fig, axes = plt.subplots(1, len(feats), figsize=(5.2*len(feats), 4.6),
                             squeeze=False)
    for j, feat in enumerate(feats):
        ax = axes[0, j]
        fi = cols.index(feat)
        xvals = Xa[:, fi]
        for code, key in contrasts:
            tau = np.ravel(strat_est.effect(Xa, T0="0_neither", T1=code))
            # Binned mean of tau across covariate deciles.
            try:
                bins = np.quantile(xvals, np.linspace(0, 1, 11))
                bins = np.unique(bins)
                idx = np.digitize(xvals, bins[1:-1])
                bx = [xvals[idx == b].mean() for b in range(len(bins)-1)
                      if np.any(idx == b)]
                by = [tau[idx == b].mean() for b in range(len(bins)-1)
                      if np.any(idx == b)]
            except Exception:
                bx, by = xvals, tau
            ax.plot(bx, by, "-o", color=ARM_COLORS[key], ms=4, lw=1.8,
                    label=ARM_LABELS[key])
        ax.axhline(0, color="k", lw=1, ls="--", alpha=0.6)
        ax.set_xlabel(feat)
        if j == 0:
            ax.set_ylabel("CATE (change in P(mRS 0-2) vs neither)")
        ax.set_title(f"Effect vs {feat}")
        if j == 0:
            ax.legend(fontsize=8)
    fig.suptitle("Shape of effect modification (binned mean CATE)",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# C1. Timing dose-response: effect of delay on P(mRS 0-2)
# ---------------------------------------------------------------------------
def plot_dose_response(timing_ests, name="C1_dose_response"):
    """
    timing_ests: dict {modality: (est, X_treated, T_minutes)}.
    Builds a cumulative dose-response relative to the earliest observed time,
    using the constant marginal (per-minute) effect integrated over delay.
    """
    fig, axes = plt.subplots(1, len(timing_ests), figsize=(6.2*len(timing_ests),
                             4.6), squeeze=False)
    for j, (mod, (est, Xtr, T)) in enumerate(timing_ests.items()):
        ax = axes[0, j]
        Xa = Xtr.to_numpy() if hasattr(Xtr, "to_numpy") else np.asarray(Xtr)
        cme = np.ravel(est.const_marginal_effect(Xa))       # per-minute, per-patient
        lb, ub = est.const_marginal_effect_interval(Xa, alpha=0.05)
        slope = float(np.mean(cme))
        slope_lb, slope_ub = float(np.mean(lb)), float(np.mean(ub))
        t0, t1 = np.percentile(T, 2), np.percentile(T, 98)
        grid = np.linspace(t0, t1, 100)
        base = grid[0]
        # Cumulative change in P vs earliest time (linear in delay).
        curve = slope * (grid - base)
        band_lb = slope_lb * (grid - base)
        band_ub = slope_ub * (grid - base)
        c = MODALITY_COLORS.get(mod, "#333333")
        ax.plot(grid, curve, color=c, lw=2.2,
                label=f"{slope*60:+.4f} / hour")
        ax.fill_between(grid, band_lb, band_ub, color=c, alpha=0.2)
        ax.axhline(0, color="k", lw=1, ls="--", alpha=0.6)
        # rug of observed treatment times
        ax.plot(T, np.full_like(T, curve.min()), "|", color=c, alpha=0.15,
                ms=8)
        ax.set_xlabel(f"Onset-to-{mod} time (minutes)")
        ax.set_ylabel("Change in P(mRS 0-2) vs earliest")
        ax.set_title(f"{mod.capitalize()} dose-response")
        ax.legend(fontsize=9)
    fig.suptitle("Timing dose-response: cost of delay on good outcome",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# C2. Per-patient timing effect (per hour of delay) distributions
# ---------------------------------------------------------------------------
def plot_timing_distributions(timing_ests, name="C2_timing_distributions"):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for mod, (est, Xtr, T) in timing_ests.items():
        Xa = Xtr.to_numpy() if hasattr(Xtr, "to_numpy") else np.asarray(Xtr)
        per_hour = np.ravel(est.const_marginal_effect(Xa)) * 60.0
        c = MODALITY_COLORS.get(mod, "#333333")
        ax.hist(per_hour, bins=40, density=True, alpha=0.45, color=c,
                label=f"{mod.capitalize()} (mean {per_hour.mean():+.4f}/hr)")
        ax.axvline(per_hour.mean(), color=c, ls=":", lw=1.4)
    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.7)
    ax.set_xlabel("Per-patient effect of +1 hour delay on P(mRS 0-2)")
    ax.set_ylabel("Density")
    ax.set_title("Heterogeneity of the timing (delay) effect across patients")
    ax.legend(fontsize=9)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# D1. Outcome-nuisance diagnostics: calibration curve + ROC
# ---------------------------------------------------------------------------
def plot_nuisance_diagnostics(X, W, y, groups, cat_idx=None, name="D1_nuisance_diagnostics"):
    from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
    from sklearn.calibration import calibration_curve
    from sklearn.metrics import roc_curve, roc_auc_score
    if cat_idx is None:
        cat_idx = list(range(W.shape[1] - len(CATEGORICAL_FEATURES), W.shape[1]))
    model_y, _ = make_nuisances(discrete_treatment=True, cat_features=cat_idx)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    proba = cross_val_predict(model_y, W, y, cv=cv, groups=groups,
                              method="predict_proba", n_jobs=-1)[:, 1]
    auc = roc_auc_score(y, proba)
    frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy="quantile")
    fpr, tpr, _ = roc_curve(y, proba)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.plot(mean_pred, frac_pos, "-o", color="#1f77b4", lw=2, label="model E[Y|W]")
    ax.set_xlabel("Mean predicted P(good outcome)")
    ax.set_ylabel("Observed fraction good")
    ax.set_title("Calibration of outcome nuisance")
    ax.legend(fontsize=9)
    ax = axes[1]
    ax.plot(fpr, tpr, color="#d62728", lw=2, label=f"ROC (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC of outcome nuisance E[Y|W]")
    ax.legend(fontsize=9)
    fig.suptitle("DML foundation check: is the outcome model trustworthy?",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# D2. Positivity / overlap: propensity distribution by arm
# ---------------------------------------------------------------------------
def plot_overlap(data, W, name="D2_overlap_positivity"):
    from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
    arm = data["treatment_group"].map(
        {"neither": 0, "thrombolysis_only": 1,
         "thrombectomy_only": 2, "both": 3}).to_numpy()
    groups = data["stroke_team"].astype(str).astype("category").cat.codes.to_numpy() \
        if "_patient_id" not in data.columns else data["_patient_id"].to_numpy()
    # Categorical columns are the last len(CATEGORICAL_FEATURES) cols of W.
    cat_idx = list(range(W.shape[1] - len(CATEGORICAL_FEATURES), W.shape[1]))
    _, model_t = make_nuisances(discrete_treatment=True, cat_features=cat_idx)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    proba = cross_val_predict(model_t, W, arm, cv=cv, groups=groups,
                              method="predict_proba", n_jobs=-1)
    classes = np.unique(arm)
    inv = {0: "neither", 1: "thrombolysis_only",
           2: "thrombectomy_only", 3: "both"}
    ncol = len(classes)
    fig, axes = plt.subplots(1, ncol, figsize=(4.2*ncol, 4.2), squeeze=False)
    for k, cls in enumerate(classes):
        ax = axes[0, k]
        key = inv[cls]
        p_k = proba[:, k]
        for a in classes:
            akey = inv[a]
            ax.hist(p_k[arm == a], bins=25, density=True, alpha=0.45,
                    color=ARM_COLORS[akey], label=ARM_LABELS[akey])
        ax.set_title(f"P(assigned = {ARM_LABELS[key]})", fontsize=10)
        ax.set_xlabel("Estimated propensity")
        if k == 0:
            ax.set_ylabel("Density")
            ax.legend(fontsize=7.5)
    fig.suptitle("Overlap / positivity: propensity by actual arm "
                 "(good overlap = distributions share support)",
                 fontsize=12.5, y=1.03)
    fig.tight_layout()
    save_and_show(fig, name)


In [ ]:
# ---------------------------------------------------------------------------
# 6. Main
# ---------------------------------------------------------------------------
def main():
    data = load_and_filter()
    data = add_treatment_columns(data)
    # Tag each original patient so bootstrap duplicates can be clustered together
    # for cluster-robust variance (partial mitigation of resampling leakage).
    data["_patient_id"] = np.arange(len(data))

    print("\nTreatment group counts (full cohort):")
    print(data["treatment_group"].value_counts())

    if BOOTSTRAP_BALANCED:
        data = balanced_bootstrap_sample(data, BOOTSTRAP_N)
        print("\nTreatment group counts (balanced bootstrap sample):")
        print(data["treatment_group"].value_counts())

    X, W, y, groups, cat_idx = build_matrices(data)

    nuisance_sanity_check(X, W, y, groups, cat_idx)

    strat_est, strat_cate = run_strategy_forest(data, X, W, y, groups, cat_idx)
    lysis_est, lysis_X, lysis_T = run_timing_forest(data, "thrombolysis")
    ectomy_est, ectomy_X, ectomy_T = run_timing_forest(data, "thrombectomy")
    timing_ests = {
        "thrombolysis": (lysis_est, lysis_X, lysis_T),
        "thrombectomy": (ectomy_est, ectomy_X, ectomy_T),
    }

    # -- Best Linear Projection summaries ------------------------------------
    print("\n" + "=" * 70)
    print("C. BEST LINEAR PROJECTION (effect modification tests)")
    print("=" * 70)
    baseline = "0_neither"
    for a, lbl in [("1_thrombolysis_only", "thrombolysis_only vs neither"),
                   ("2_thrombectomy_only", "thrombectomy_only vs neither"),
                   ("3_both", "both vs neither")]:
        best_linear_projection(strat_est, X, lbl, list(X.columns),
                               T0=baseline, T1=a)
    # Timing forests: BLP of the per-minute delay effect on X (treated subgroup).
    for est, mod in [(lysis_est, "thrombolysis"), (ectomy_est, "thrombectomy")]:
        Xt = data[data[(TIME_THROMBOLYSIS if mod == "thrombolysis"
                        else TIME_THROMBECTOMY)].notna()][HETERO_FEATURES]
        Xt = Xt.fillna(Xt.median(numeric_only=True))
        best_linear_projection(est, Xt, f"{mod} delay (per-minute effect)",
                               HETERO_FEATURES)

    # -- Visualization suite (inline + saved to output/) ---------------
    print("\n" + "=" * 70)
    print("E. VISUALIZATION SUITE -> output/")
    print("=" * 70)
    # A. Strategy
    plot_ate_forest(strat_est, X)
    plot_cate_distributions(strat_cate)
    plot_shap_summary(strat_est, X)
    # B. Effect modification
    plot_blp_forest(strat_est, X)
    plot_cate_vs_covariate(strat_est, X)
    # C. Timing
    plot_dose_response(timing_ests)
    plot_timing_distributions(timing_ests)
    # D. Diagnostics
    plot_nuisance_diagnostics(X, W, y, groups, cat_idx)
    plot_overlap(data, W)

    # ---------------------------------------------------------------------
    # FULL per-patient export: all model inputs + treatment effects on BOTH
    # the probability (risk-difference) and log-odds scales, for every arm.
    # ---------------------------------------------------------------------
    from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold

    # Baseline risk p0 = out-of-fold E[Y|W] from the SAME nuisance the forest
    # uses. Effects are expressed relative to this per-patient baseline.
    _model_y, _ = make_nuisances(discrete_treatment=True, cat_features=cat_idx)
    _cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p0 = cross_val_predict(_model_y, W, y, cv=_cv, groups=groups,
                           method="predict_proba", n_jobs=-1)[:, 1]
    eps = 1e-6
    p0c = np.clip(p0, eps, 1 - eps)
    logit_p0 = np.log(p0c / (1 - p0c))

    # ----- (1) all model INPUT data used -----
    out = pd.DataFrame(index=data.index)
    for col in HETERO_FEATURES:            # heterogeneity features (X)
        out[col] = data[col].to_numpy()
    for col in CATEGORICAL_FEATURES:       # categorical nuisance controls in W
        out[col] = data[col].to_numpy()
    out["treatment_group"] = data["treatment_group"].to_numpy()
    out["got_thrombolysis"] = data["got_thrombolysis"].to_numpy()
    out["got_thrombectomy"] = data["got_thrombectomy"].to_numpy()
    out[TIME_THROMBOLYSIS] = data[TIME_THROMBOLYSIS].to_numpy()
    out[TIME_THROMBECTOMY] = data[TIME_THROMBECTOMY].to_numpy()
    out["discharge_disability"] = data["discharge_disability"].to_numpy()
    out["y_good_outcome"] = y              # binary target actually modelled
    out["mrs_target"] = MRS_TARGET         # provenance of the threshold
    out["cluster_group"] = groups          # cross-fitting cluster id
    out["baseline_risk_p0"] = p0           # modelled per-patient baseline risk
    # Predicted OUTCOME LEVEL under "neither" (untreated) on the prob scale.
    out["pred_outcome_untreated_p"] = p0

    # ----- (2)+(3) effects for EACH arm on P and log-odds scales -----
    arm_to_col = {"thrombolysis_only": "1_thrombolysis_only",
                  "thrombectomy_only": "2_thrombectomy_only",
                  "both": "3_both"}
    for a, cate in strat_cate.items():
        arm = a[2:]                        # strip "1_" etc.
        cate = np.ravel(cate)
        out[f"cate_prob_{arm}_vs_neither"] = cate
        p1c = np.clip(p0 + cate, eps, 1 - eps)
        out[f"cate_logodds_{arm}_vs_neither"] = np.log(p1c / (1 - p1c)) - logit_p0
        out[f"odds_ratio_{arm}_vs_neither"] = np.exp(
            out[f"cate_logodds_{arm}_vs_neither"].to_numpy())
        # Predicted OUTCOME LEVEL if given this arm = p0 + CATE (prob scale).
        out[f"pred_outcome_treated_p_{arm}"] = np.clip(p0 + cate, eps, 1 - eps)

    # ----- (4) matched-arm effect (arm each patient ACTUALLY received) -----
    matched_cate = np.zeros(len(data))
    grp_arr = data["treatment_group"].to_numpy()
    for i, g in enumerate(grp_arr):
        code = arm_to_col.get(g)
        matched_cate[i] = np.ravel(strat_cate[code])[i] if code else 0.0
    p1c = np.clip(p0 + matched_cate, eps, 1 - eps)
    out["cate_matched_prob"] = matched_cate
    out["cate_matched_logodds"] = np.log(p1c / (1 - p1c)) - logit_p0
    out["odds_ratio_matched"] = np.exp(out["cate_matched_logodds"].to_numpy())
    # Predicted OUTCOME LEVELS for the arm each patient ACTUALLY received:
    # untreated (neither) vs treated (received arm). For "neither" patients the
    # two are equal (matched_cate == 0). Absolute P(good outcome).
    out["pred_outcome_matched_untreated_p"] = p0
    out["pred_outcome_matched_treated_p"] = np.clip(p0 + matched_cate, eps, 1 - eps)

    print(f"\n[export] matched-arm effect: "
          f"P mean={matched_cate.mean():+.4f} sd={matched_cate.std():.4f} | "
          f"log-odds mean={out['cate_matched_logodds'].mean():+.4f} "
          f"(OR mean={out['odds_ratio_matched'].mean():.3f})")

    cate_csv = out_path("causal_forest_effects_by_patient.csv")
    out.to_csv(cate_csv, index=False)
    print(f"[export] saved {out.shape[0]} patients x {out.shape[1]} columns "
          f"(all model data + P & log-odds effects) -> {cate_csv}")
    # ---------------------------------------------------------------------
    # Persist fitted MODELS (+ SHAP values) as pickled files in output/.
    # Uses joblib (robust for sklearn/EconML objects) with a pickle fallback.
    # ---------------------------------------------------------------------
    import pickle
    try:
        import joblib
        _dump = lambda obj, path: joblib.dump(obj, path)
    except Exception:
        def _dump(obj, path):
            with open(path, "wb") as fh:
                pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)

    # Compute SHAP values for the strategy CATE model so they can be saved
    # alongside the models (plot_shap_summary computes them internally but
    # discards them; here we keep them for reuse).
    shap_values = None
    try:
        shap_values = strat_est.shap_values(
            X.to_numpy(), feature_names=list(X.columns))
        print("[models] computed SHAP values for strategy model")
    except Exception as exc:
        print(f"[models] SHAP computation failed, saving None: {exc}")

    models_bundle = {
        "strategy_forest": strat_est,
        "timing_thrombolysis": lysis_est,
        "timing_thrombectomy": ectomy_est,
        "shap_values_strategy": shap_values,
        "metadata": {
            "mrs_target": MRS_TARGET,
            "hetero_features": list(HETERO_FEATURES),
            "categorical_features": list(CATEGORICAL_FEATURES),
            "categorical_indices_in_W": cat_idx,
            "random_state": RANDOM_STATE,
            "bootstrap_balanced": BOOTSTRAP_BALANCED,
            "n_patients": int(len(data)),
            "saved_at": pd.Timestamp.now().isoformat(),
        },
    }

    # One combined bundle (everything together)...
    bundle_path = out_path("fitted_models_bundle.pkl")
    _dump(models_bundle, bundle_path)
    print(f"[models] saved combined bundle -> {bundle_path}")

    # ...and individual files for convenience.
    for key, obj in [("strategy_forest", strat_est),
                     ("timing_thrombolysis", lysis_est),
                     ("timing_thrombectomy", ectomy_est),
                     ("shap_values_strategy", shap_values)]:
        p = out_path(f"{key}.pkl")
        _dump(obj, p)
        print(f"[models] saved {key} -> {p}")

    print("\nDone.")


In [ ]:
if __name__ == "__main__":
    main()